# MODEL: Evaluation and Baseline Performance

In [6]:
import subprocess
import sys
import os

def get_repo_root():
    return subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).decode('utf-8').strip()
repo_root = get_repo_root()
print(repo_root)

src_path = os.path.join(repo_root, 'src')
# Add src_path to sys.path if not already present
if src_path not in sys.path:
    sys.path.insert(0, src_path)

/home/sagemaker-user/aai-540-su25-group4


In [16]:
# try importing src/utils
from utils.utils import parse_s3_uri
from utils.utils import generate_manifest_file


In [18]:
import boto3
import sagemaker
from sagemaker import get_execution_role
import pandas as pd
import json
from sagemaker.transformer import Transformer


role = get_execution_role()
print(role)

region = boto3.Session().region_name

s3_client = boto3.client("s3")
sm_client = boto3.client("sagemaker")

sess = sagemaker.Session()

# project bucket
bucket_name = "aai-540-data"

# sagemaker model name
sagemaker_model_name = "image-classification-2025-06-19-18-37-41-534"

# provide validation and test s3 folders,
s3_validation_path = f"s3://{bucket_name}/dev_split/validation"
val_key = "val-meta.csv"

s3_test_path = f"s3://{bucket_name}/dev_split/test"
test_key = "test-meta.csv"

# provide s3 full path to label_mapping.json use during training
s3_label_map_uri = f"s3://{bucket_name}/dev_split/label_mapping.json"


arn:aws:iam::324183265896:role/service-role/AmazonSageMaker-ExecutionRole-20250604T045982


In [19]:
# Generate Manifest File for Validation and Test Meta CSVs for Evaluation Via Batch Transform
generate_manifest_file(s3_input_csv = f"{s3_validation_path}/{val_key}", s3_images_loc = f"s3://{bucket_name}/cct_resized/")
generate_manifest_file(s3_input_csv = f"{s3_test_path}/{test_key}", s3_images_loc = f"s3://{bucket_name}/cct_resized/")


File uploaded to s3://aai-540-data/dev_split/validation/val-meta.manifest
File uploaded to s3://aai-540-data/dev_split/test/test-meta.manifest


In [21]:
# Transform the Validation Set First

s3_transform_manifest = f"{s3_validation_path}/val-meta.manifest"
s3_transform_out = f"{s3_validation_path}/batch_transform_out"

# initialize Tranformer
transformer = Transformer(
    model_name = sagemaker_model_name,
    instance_count=1,  # Number of instances
    instance_type="ml.g4dn.xlarge",  # Instance type
    output_path= s3_transform_out,  # Predictions output
    max_payload=10,  # Max payload size (MB)
    strategy="MultiRecord" , # for faster processing, but in real world, instance type can be ml.m5.xlarge and single record strategy is ok
    max_concurrent_transforms=10,
    sagemaker_session=sess,

    accept = 'txt/csv', # so output is generated in single file
    assemble_with='Line', # new line is generated for each prediction

)

In [23]:
# Transform the Validation Set

# batch transform images in manifest file
transformer.transform(
    data=s3_transform_manifest,
    data_type='ManifestFile', # provide list of s3uris of objects to be batch transformed
    content_type='application/x-image', 
    split_type='None', # because each object is an image file to be processed, no splitting needed
    logs=True,
    wait=True
)


INFO:sagemaker:Creating transform job with name: image-classification-2025-06-22-13-12-48-727


...................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
Nvidia gpu devices, drivers and cuda toolkit versions (only available on hosts with GPU):
Sun Jun 22 13:18:32 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.256.02   Driver Version: 470.256.02   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:1E.0 Off |                    0 |
| N/A   26C    P8    11W /  70W |      0MiB / 15109MiB |      0%      Default |
|                        

In [31]:
# Evaluate the Prediction Results via ScriptProcessing
from sagemaker.processing import ScriptProcessor, ProcessingInput, ProcessingOutput

image_uri = sagemaker.image_uris.retrieve(
    framework='sklearn',        # or 'xgboost', 'pytorch', etc.
    region=region,
    version='1.2-1',            # Specify the version you need
    py_version='py3',           # Specify Python version if required
       # Use 'processing' for processing jobs
)

print(image_uri)

# Define your processing container (can use a built-in or custom image)
script_processor = ScriptProcessor(
    command=['python3'],
    image_uri=image_uri,  # e.g., a scikit-learn or custom image
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    
)


INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


In [32]:

# Run evaluation script
s3_evaluation_out = f"{s3_validation_path}/evaluation"
s3_true_meta_uri = f"{s3_validation_path}/{val_key}"

script_processor.run(
    code='../../src/evaluation/evaluate.py',  # Your processing script
    inputs=[
        # S3 location of batch transform predictions files
        ProcessingInput(
            source=s3_transform_out,       # S3 bucket with predictions
            destination='/opt/ml/processing/input_predictions'        # Where the script will read input
        ),
        
        # S3 location of the ground truth labels for the images in this set
        ProcessingInput(
            source=s3_true_meta_uri,
            destination='/opt/ml/processing/true_labels'
        ),

        # Label Mapping
        ProcessingInput(
            source=s3_label_map_uri,
            destination='/opt/ml/processing/label_mapping'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',           # Where the script will write output
            destination=s3_evaluation_out    # S3 bucket to store results
        )
    ]
)

INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2025-06-22-14-11-22-529


...........[2025-06-22 14:13:14.822629] Script has started.
INFO:root:Script has started at 2025-06-22 14:13:14.822659.
Number of files in /opt/ml/processing/input_predictions: 6048
Processed 3 prediction files so far, here is the dataframe
                                     filename                                         prediction
0  5971fad6-23d2-11e8-a6a3-ec086b02610b_0.jpg  [1.4443931206642438e-07, 7.355709097378593e-11...
1  59dc24e9-23d2-11e8-a6a3-ec086b02610b_0.jpg  [9.587807170419183e-08, 6.917158401620328e-09,...
2  5968c1b0-23d2-11e8-a6a3-ec086b02610b_0.jpg  [0.00269423238933, 3.909136648871936e-05, 2.80...
                                     filename                                         prediction
0  5971fad6-23d2-11e8-a6a3-ec086b02610b_0.jpg  [1.4443931206642438e-07, 7.355709097378593e-11...
1  59dc24e9-23d2-11e8-a6a3-ec086b02610b_0.jpg  [9.587807170419183e-08, 6.917158401620328e-09,...
2  5968c1b0-23d2-11e8-a6a3-ec086b02610b_0.jpg  [0.00269423238933, 3.909136648871

----
### Model Comparison (on Validation Set) & Selection

-----
### Check Performance on Test Set of the selected Model

In [33]:
# Transform the Test Set
s3_transform_manifest = f"{s3_test_path}/test-meta.manifest"
s3_transform_out = f"{s3_test_path}/batch_transform_out"

# initialize Tranformer
transformer1 = Transformer(
    model_name = sagemaker_model_name,
    instance_count=1,  # Number of instances
    instance_type="ml.g4dn.xlarge",  # Instance type
    output_path= s3_transform_out,  # Predictions output
    max_payload=10,  # Max payload size (MB)
    strategy="MultiRecord" , # for faster processing, but in real world, instance type can be ml.m5.xlarge and single record strategy is ok
    max_concurrent_transforms=10,
    sagemaker_session=sess,

    accept = 'txt/csv', # so output is generated in single file
    assemble_with='Line', # new line is generated for each prediction

)

# batch transform images in manifest file
transformer1.transform(
    data=s3_transform_manifest,
    data_type='ManifestFile', # provide list of s3uris of objects to be batch transformed
    content_type='application/x-image', 
    split_type='None', # because each object is an image file to be processed, no splitting needed
    logs=True,
    wait=True
)


INFO:sagemaker:Creating transform job with name: image-classification-2025-06-22-14-19-58-779


....................................Docker entrypoint called with argument(s): serve
Running default environment configuration script
Nvidia gpu devices, drivers and cuda toolkit versions (only available on hosts with GPU):
Sun Jun 22 14:25:56 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.256.02   Driver Version: 470.256.02   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla T4            On   | 00000000:00:1E.0 Off |                    0 |
| N/A   34C    P8     9W /  70W |      0MiB / 15109MiB |      0%      Default |
|                       

In [34]:
# Run Evaluation Script to evaluate all metrics

image_uri = sagemaker.image_uris.retrieve(
    framework='sklearn',        # or 'xgboost', 'pytorch', etc.
    region=region,
    version='1.2-1',            # Specify the version you need
    py_version='py3',           # Specify Python version if required
       # Use 'processing' for processing jobs
)
print(image_uri)
# Define your processing container (can use a built-in or custom image)
script_processor1 = ScriptProcessor(
    command=['python3'],
    image_uri=image_uri,  # e.g., a scikit-learn or custom image
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    
)

# Run the processing job on the validation prediction set
s3_evaluation_out = f"{s3_test_path}/evaluation"
s3_true_meta_uri = f"{s3_test_path}/{test_key}"

script_processor1.run(
    code='../../src/evaluation/evaluate.py',  # Your processing script
    inputs=[
        # S3 location of batch transform predictions files
        ProcessingInput(
            source=s3_transform_out,       # S3 bucket with predictions
            destination='/opt/ml/processing/input_predictions'        # Where the script will read input in local container
        ),
        
        # S3 location of the ground truth labels for the images in this set
        ProcessingInput(
            source=s3_true_meta_uri,
            destination='/opt/ml/processing/true_labels'
        ),

        # Label Mapping
        ProcessingInput(
            source=s3_label_map_uri,
            destination='/opt/ml/processing/label_mapping'
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',           # Where the script will write output files in local container
            destination=s3_evaluation_out    # S3 bucket to store results
        )
    ]
)

INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.


683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-scikit-learn:1.2-1-cpu-py3


INFO:sagemaker:Creating processing-job with name sagemaker-scikit-learn-2025-06-22-14-30-07-960


.............[2025-06-22 14:32:13.275552] Script has started.
INFO:root:Script has started at 2025-06-22 14:32:13.275586.
Number of files in /opt/ml/processing/input_predictions: 7932
pred probs df shape:(7932, 2)
 (step 1) pred probs df shape: (7932, 2)
 (step 2) True Labels data shape: (7932, 8)
 (step3) Merged data shape : (7932, 9)
 (step4) Merged data shape with preds: (7932, 11)
 (step5) METRICS CALCULATION
 (step5.1) Class-Restricted metrics - Evaluate only on labels available during training
 (step5.1) class_restriced df shape : (7804, 11)
/miniconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/miniconda3/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill

------

### Performance Summary on Validation and Test Sets

In [40]:
# validation set
s3_metrics_uri = f"{s3_validation_path}/evaluation/multiclass_metrics.json"
s3_class_report_uri = f"{s3_validation_path}/evaluation/restricted_class_report.csv"
val_metrics = pd.read_json(s3_metrics_uri)
val_class_report = pd.read_csv(s3_class_report_uri)
display(val_metrics)
display(val_class_report)

,multiclass_classification_metrics
accuracy,"{'value': 0.8903769841269841, 'standard_deviat..."
f1-weighted,"{'value': 0.892783336735068, 'standard_deviati..."
novelty_ratio,"{'value': 0.074074074074074, 'standard_deviati..."


,Unnamed: 0,precision,recall,f1-score,support
0,bird,0.322034,0.730769,0.447059,26.000000
1,bobcat,0.813525,0.874449,0.842887,454.000000
2,car,1.000000,1.000000,1.000000,424.000000
3,cat,0.754955,0.819961,0.786116,511.000000
4,coyote,0.877934,0.839820,0.858454,668.000000
5,deer,1.000000,0.800000,0.888889,5.000000
6,dog,0.861446,0.752632,0.803371,380.000000
7,empty,0.000000,0.000000,0.000000,13.000000
8,opossum,0.990533,0.918771,0.953303,1822.000000
9,rabbit,0.818471,0.948339,0.878632,271.000000


In [39]:
# test
s3_metrics_uri = f"{s3_test_path}/evaluation/multiclass_metrics.json"
s3_class_report_uri = f"{s3_test_path}/evaluation/restricted_class_report.csv"
test_metrics = pd.read_json(s3_metrics_uri)
test_class_report = pd.read_csv(s3_class_report_uri)
display(test_metrics)
display(test_class_report)

,multiclass_classification_metrics
accuracy,"{'value': 0.845079446437724, 'standard_deviati..."
f1-weighted,"{'value': 0.8479535389038411, 'standard_deviat..."
novelty_ratio,"{'value': 0.09934442763489601, 'standard_devia..."


,Unnamed: 0,precision,recall,f1-score,support
0,badger,0.000000,0.000000,0.000000,4.000000
1,bird,0.293478,0.500000,0.369863,54.000000
2,bobcat,0.751189,0.833040,0.790000,569.000000
3,car,1.000000,0.988235,0.994083,85.000000
4,cat,0.683429,0.722222,0.702290,828.000000
5,coyote,0.853866,0.838198,0.845960,1199.000000
6,deer,1.000000,0.470588,0.640000,17.000000
7,dog,0.624658,0.577215,0.600000,395.000000
8,empty,0.000000,0.000000,0.000000,8.000000
9,opossum,0.991835,0.903622,0.945676,3092.000000


In [41]:
!pwd

/home/sagemaker-user/aai-540-su25-group4/experiments/geoffrey
